# Data Science for Business - Multilayer Perceptron (MLP) on Ames Housing with Scikit-learn

*Session 8 · ISLP 10.1–10.2 · 10.7 Fitting a Neural Network*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import LassoCV
from sklearn.neural_network import MLPRegressor

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

Ask a home buyer to describe their dream house, and they probably won't begin with the height of the basement ceiling or the proximity to an east-west railroad. But this dataset proves that much more influences price negotiations than the number of bedrooms or a white-picket fence. With 76 explanatory variables describing (almost) every aspect of residential homes in Ames, Iowa, this dataset challenges you to predict the final price of each home. More: <https://www.kaggle.com/c/house-prices-advanced-regression-techniques>

## Load data

Load training data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_08/ameshousing.csv')

In [ ]:
data.head()

## Prepare data

First, we will remove some columns that are not useful for our task.

In [ ]:
data = data.drop(['YrSold', 'MoSold', 'SaleCondition', 'SaleType'], axis=1)

`MSSubClass` (type of dwelling) is stored as a number, but its values are codes for categories (e.g., 20 = 1-story house built 1946 or later, 60 = 2-story house built 1946 or later; see the data description). We therefore add the prefix `MSSC` to all values, which turns the variable into a string, so that it is treated as categorical.

In [ ]:
data["MSSubClass"] = "MSSC" + data["MSSubClass"].astype(str)

Next, we will split the data into predictors (*X*) and response (*y*) and into training (*X_train, y_train*) and test (*X_test, y_test*) sets. We use the same split as in Session 5, so that we can compare the neural network with the lasso.

In [ ]:
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Finally, we will prepare the predictors for the models. It is important to use only information from the training set for this (e.g., the categories and the means and standard deviations), and then mechanistically repeat these steps on the test set. A scikit-learn pipeline takes care of this for us.

Typically, the preparation depends strongly on the datatype of the variables. Hence, we will first determine which variables are categorical and which are numerical. Subsequently, we will transform these variables separately.

In [ ]:
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns

The categorical variables must be transformed into numerical representations, e.g., by one-hot encoding them. `handle_unknown='ignore'` encodes categories that appear only in the test set as all zeros instead of raising an error.

In [ ]:
onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

The numerical variables will be standardized, that is, we will subtract the mean and divide by the standard deviation. Neural networks are trained by gradient descent, which works much better when all inputs are on a similar scale; the lasso needs standardized predictors as well, because its penalty depends on the scale of the coefficients (ISLP 6.2).

In [ ]:
scaler = StandardScaler()

A `ColumnTransformer` applies each transformer to its own columns and puts the results side by side. `.set_output(transform='pandas')` returns a DataFrame, so the feature names stay readable.

In [ ]:
preprocessor = ColumnTransformer(
                transformers=[
                        ('num', scaler, numerical_features),
                        ('cat', onehot, categorical_features)
                    ], verbose_feature_names_out=False
                ).set_output(transform='pandas')

We do not apply the preprocessor to the data by hand. Instead, we combine it with each model in a `Pipeline`: a chain of steps that behaves like a single model. `fit` runs all steps in order (the encoder and scaler learn from the training data, then the model is trained), and `predict` applies the same steps to new data, e.g., the test set. This gives us one object to fit and predict, and if we cross-validate a pipeline (e.g., with `GridSearchCV`), the preprocessing is refitted in every fold on the training folds only, so no information leaks from the held-out fold.

The response needs scaling, too: sale prices are around 180,000, so the squared-error loss on the raw prices is in the order of $10^{10}$ and the gradients are poorly scaled. Gradient-based training works best when inputs and output are on a similar scale, so we standardize the response with the mean and standard deviation of the training set and transform the predictions back to dollars at the end. The PyTorch notebook (`ames_mlp_pytorch.ipynb`) does exactly the same.

We do this as an explicit, visible step outside the pipeline: the pipeline prepares the predictors, while we take care of the response ourselves. (scikit-learn's `TransformedTargetRegressor` could wrap this around the pipeline, but the explicit version shows every step and gives us direct access to the network's training curve below.)

In [ ]:
y_mean, y_std = y_train.mean(), y_train.std()

y_train_scaled = (y_train - y_mean) / y_std

## Baseline: lasso

We first fit a linear model as a baseline. We use the lasso (ISLP 6.2.2), with λ chosen by 5-fold cross-validation, on the same predictors and, like the network, with the standardized price as response. The pipeline has two steps, which we name `preprocess` and `model`; we fit it on the raw training predictors.

In [ ]:
model_lasso = Pipeline([
                ('preprocess', preprocessor),
                ('model', LassoCV(cv=5, random_state=42))
                ])
model_lasso.fit(X_train, y_train_scaled)

In [ ]:
pred_test_lasso = model_lasso.predict(X_test) * y_std + y_mean
rmse_test_lasso = root_mean_squared_error(y_test, pred_test_lasso)
print('Lasso R2 on test set:', round(r2_score(y_test, pred_test_lasso), 2))
print('Lasso RMSE on test set:', round(rmse_test_lasso, 2))

## Neural network

In the following, we will create a feedforward neural network with three hidden layers (ISLP 10.2) comprising 512, 256 and 128 hidden units, the same architecture as in the PyTorch notebook. Each hidden unit computes a linear combination of its inputs followed by the ReLU activation function $g$, e.g., $A_k = g(w_{k0} + \sum_j w_{kj}X_j)$ in the first hidden layer; the output is a linear combination of the units of the last hidden layer (ISLP 10.1, equations 10.1 and 10.2).

The network is trained with Adam, a popular variant of stochastic gradient descent (ISLP 10.7.2), with learning rate ρ = 0.001 (`learning_rate_init`). One epoch is one pass over the training data; we allow at most 1000 epochs. With `early_stopping=True`, scikit-learn sets aside 10% of the training data as a validation set and stops when the validation score has not improved for 10 epochs (early stopping, ISLP 10.7.2). scikit-learn's `MLPRegressor` has no dropout; its `alpha`, the strength of an L2 (ridge) penalty on the weights (yet another meaning of "alpha"), plays a similar regularising role, while the PyTorch notebook uses dropout. As for the lasso, we put the preprocessor and the network into one pipeline.

In [ ]:
model_mlp = Pipeline([
                ('preprocess', preprocessor),
                ('model', MLPRegressor(hidden_layer_sizes=(512, 256, 128), activation='relu', solver='adam', learning_rate_init=0.001, alpha=0.0001,
                                       max_iter=1000, early_stopping=True, validation_fraction=0.1, random_state=42))
                ])

In [ ]:
model_mlp.fit(X_train, y_train_scaled)

Let's look at the training process: the training loss (squared error on the standardized scale) and the R² on the internal validation set per epoch. These are stored in the fitted network, i.e., the `model` step of the pipeline.

In [ ]:
mlp = model_mlp.named_steps['model']
print('Number of epochs:', mlp.n_iter_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(mlp.loss_curve_)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training loss')
axes[1].plot(mlp.validation_scores_)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation R2')
plt.tight_layout()
plt.show()

Now we evaluate the network on the training and test set. The network predicts standardized prices, so we transform the predictions back to dollars: multiply by the standard deviation and add the mean.

In [ ]:
# Training data
pred_train = model_mlp.predict(X_train) * y_std + y_mean
r2_train = r2_score(y_train, pred_train)
rmse_train = root_mean_squared_error(y_train, pred_train)
print('R2 on training set:', round(r2_train, 2))
print('RMSE on training set:', round(rmse_train, 2))

print("===")

# Test data
pred_test = model_mlp.predict(X_test) * y_std + y_mean
r2_test = r2_score(y_test, pred_test)
rmse_test = root_mean_squared_error(y_test, pred_test)
print('R2 on test set:', round(r2_test, 2))
print('RMSE on test set:', round(rmse_test, 2))

How to read these numbers: a big gap between training and test performance indicates overfitting; similar but poor values indicate underfitting or an unfinished optimisation.

Finally, let's compare the network with the lasso baseline.

In [ ]:
pd.DataFrame({'Model': ['Lasso', 'MLP'], 'Test RMSE': [rmse_test_lasso, rmse_test]}).round(0)

Here, the network beats the lasso, though not by a wide margin. ISLP 10.6 ("When to Use Deep Learning") cautions that neural networks shine when the sample size is large and the signal is complex (images, text, ...), while for small tabular datasets like ours (about 2,300 training observations), simpler models often perform nearly as well and are easier to interpret.

The PyTorch notebook (`ames_mlp_pytorch.ipynb`) fits the same model, with the same response treatment and architecture, but writes the training loop by hand; compare its test RMSE with the table above.